In [61]:

import pandas as pd

df = pd.read_csv('gk_qna_dataset.csv')

df.head()

,question,answer
0,"What is the capital of ""France""?",Paris
1,"What is the capital of ""Germany""?",Berlin
2,"What is the capital of ""Italy""?",Rome
3,"What is the capital of ""Spain""?",Madrid
4,"What is the capital of ""Japan""?",Tokyo


In [62]:
import re

def tokenize(text):
    # lowercase
    text = text.lower()
    # remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # remove quotes
    text = re.sub(r'[\'"]', '', text)
    tokens = text.split()
    
    return tokens
    




In [63]:
vocab = {'<UNK>': 0}

def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])
    merged_tokens = tokenized_question + tokenized_answer    
    for token in merged_tokens:
        if token not in vocab:
            vocab[token] = len(vocab)
    
df.apply(build_vocab, axis=1)
print(vocab)

{'<UNK>': 0, 'what': 1, 'is': 2, 'the': 3, 'capital': 4, 'of': 5, 'france': 6, 'paris': 7, 'germany': 8, 'berlin': 9, 'italy': 10, 'rome': 11, 'spain': 12, 'madrid': 13, 'japan': 14, 'tokyo': 15, 'canada': 16, 'ottawa': 17, 'brazil': 18, 'brasilia': 19, 'australia': 20, 'canberra': 21, 'india': 22, 'new': 23, 'delhi': 24, 'china': 25, 'beijing': 26, 'russia': 27, 'moscow': 28, 'united': 29, 'states': 30, 'washington': 31, 'dc': 32, 'mexico': 33, 'city': 34, 'egypt': 35, 'cairo': 36, 'turkey': 37, 'ankara': 38, 'argentina': 39, 'buenos': 40, 'aires': 41, 'south': 42, 'korea': 43, 'seoul': 44, 'indonesia': 45, 'jakarta': 46, 'pakistan': 47, 'islamabad': 48, 'bangladesh': 49, 'dhaka': 50, 'nepal': 51, 'kathmandu': 52, 'sri': 53, 'lanka': 54, 'colombo': 55, 'thailand': 56, 'bangkok': 57, 'malaysia': 58, 'kuala': 59, 'lumpur': 60, 'vietnam': 61, 'hanoi': 62, 'uae': 63, 'abu': 64, 'dhabi': 65, 'iran': 66, 'tehran': 67, 'iraq': 68, 'baghdad': 69, 'saudi': 70, 'arabia': 71, 'riyadh': 72, 'nige

In [64]:

def text_to_indices(text,vocab):
    indexed_text = []
    
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
            
    return indexed_text

text_to_indices("What is the capital of BD?", vocab)
        

[1, 2, 3, 4, 5, 0]

In [65]:
import torch
from torch.utils.data import Dataset, DataLoader

class QnADataset(Dataset):
    def __init__(self, df, vocab):
        self.data = df
        self.vocab = vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        question = self.data.iloc[idx]["question"]
        answer = self.data.iloc[idx]["answer"]

        question_indices = text_to_indices(question, self.vocab)
        answer_indices = text_to_indices(answer, self.vocab)

        return torch.tensor(question_indices), torch.tensor(answer_indices)

In [66]:
dataset = QnADataset(df, vocab)
dataset[1]

(tensor([1, 2, 3, 4, 5, 8]), tensor([9]))

In [67]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

for questions, answers in dataloader:
    print("Batch of questions:", questions)
    print("Batch of answers:", answers)
    break  # Just to show one batch

Batch of questions: tensor([[ 79, 105,   3, 106, 107,  95, 232, 233]])
Batch of answers: tensor([[108, 109, 110]])
